# Criba de motores TTS — OmniVoice

`k2-fsa/OmniVoice` en modo clonación (referencia + su transcripción), `language="es"`, float16 y el resto de parámetros por defecto.

**GPU:** T4. **Tiempo estimado:** unos 30 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **T4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q omnivoice==0.2.1

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencias
import os, sys, subprocess
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

print(sys.version.split()[0], "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                                  capture_output=True, text=True).stdout.strip())
comun.montar_drive()
ITEMS, SILENCIO_MS = comun.cargar_items(None)
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Cargar el modelo
import torch
from omnivoice import OmniVoice
comun.guardar_entorno("omnivoice", ["omnivoice", "torch", "transformers"])
modelo = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
SR = modelo.sampling_rate or 24000

def sintetizar(texto, ref):
    audios = modelo.generate(text=texto, language="es", ref_audio=ref["wav"], ref_text=ref["texto"])
    return audios[0], SR
print("OK, sale a", SR, "Hz")

In [ ]:
# 4. Generar
comun.generar("omnivoice", sintetizar, ITEMS, REFS, SILENCIO_MS,
              params={"modelo": "k2-fsa/OmniVoice", "language": "es", "dtype": "float16"})

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['omnivoice']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()